# HAMIC — TPU evaluation

List runs with a terminal `COMPLETE.json`, pick one or more, then re-score the same parameters
under token, span and PMI masking. This notebook is read-only: it downloads committed
checkpoints and the verified corpus, and never publishes or rewrites a run.

PMI mining scans the train split once per evaluation session. Set `RUN_IDS = []` to use the first
finished run. The token is read as `HF_KEY` (TPU only).

In [ ]:
HAMIC_REF = "main"
HUB_REPO = "HaianCao/hamic-runs"
DATA_REPO = "HaianCao/hamic-data"
XET_HIGH_PERFORMANCE = True

# Empty = first finished run. Use exact run ids from the discovery cell.
RUN_IDS = []
VIEWS = ("token", "span")
SEQUENCES = 2048
MASK_RATE = 0.15

In [ ]:
import json
import os
import shutil
import sys
import tarfile
import tempfile
import urllib.request
from pathlib import Path

os.environ["HF_XET_HIGH_PERFORMANCE"] = "1" if XET_HIGH_PERFORMANCE else "0"
work_root = Path(tempfile.gettempdir()) / "hamic-tpu"
source_dir = work_root / "source"
stamp = work_root / "source-ref.json"
repo = "https://github.com/HaianCao/HAMIC-LM"
with urllib.request.urlopen(
    f"https://api.github.com/repos/HaianCao/HAMIC-LM/commits/{HAMIC_REF}",
    timeout=30,
) as response:
    commit = json.load(response)["sha"]
stamp_data = json.loads(stamp.read_text()) if stamp.is_file() else None
if not isinstance(stamp_data, dict):
    stamp_data = None
if (
    not source_dir.is_dir()
    or stamp_data is None
    or stamp_data.get("ref") != HAMIC_REF
    or stamp_data.get("commit") != commit
):
    archive = work_root / "source.tar.gz"
    work_root.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(f"{repo}/archive/{commit}.tar.gz", archive)
    unpack = work_root / "unpack"
    shutil.rmtree(unpack, ignore_errors=True)
    unpack.mkdir()
    with tarfile.open(archive) as tar:
        try:
            tar.extractall(unpack, filter="data")
        except TypeError:
            tar.extractall(unpack)
    (top,) = unpack.iterdir()
    shutil.rmtree(source_dir, ignore_errors=True)
    shutil.move(str(top), source_dir)
    shutil.rmtree(unpack, ignore_errors=True)
    archive.unlink()
    stamp.write_text(json.dumps({"ref": HAMIC_REF, "commit": commit}))
sys.path.insert(0, str(source_dir / "src"))
if "hamic_jax" in sys.modules:
    raise RuntimeError(
        "An older hamic_jax is already imported. Restart the kernel and run this notebook again."
    )
import hamic_jax

if Path(hamic_jax.__file__).resolve().parent != (source_dir / "src" / "hamic_jax").resolve():
    raise RuntimeError("hamic_jax was imported from outside the pinned source tree")
print(f"Source {HAMIC_REF} resolved to {commit[:12]} · work root {work_root}")
print("The wheel excludes src/hamic_jax, so this notebook imports the pinned checkout.")

In [ ]:
from hamic_jax.dependencies import ensure_dependencies

installed = ensure_dependencies()
print("Runtime dependencies are available." if not installed else f"Installed: {', '.join(installed)}")

In [ ]:
import jax

backend = jax.default_backend()
if backend != "tpu":
    raise RuntimeError(
        f"No TPU attached (jax.default_backend()={backend!r}). Attach the TPU and rerun."
    )
devices = jax.devices()
print(
    f"JAX {jax.__version__} · backend {backend} · {len(devices)} device(s) "
    f"{devices[0].device_kind} · code {commit[:12]}"
)

In [ ]:
import getpass

hub_token = os.environ.get("HF_KEY", "").strip()
if not hub_token:
    hub_token = getpass.getpass("Hugging Face token (HF_KEY): ").strip()
if not hub_token:
    raise RuntimeError("Enter a Hugging Face token in HF_KEY or at the prompt.")

In [ ]:
from hamic_jax.checkpoint import HuggingFaceObjectStore

run_store = HuggingFaceObjectStore(HUB_REPO, private=True, token=hub_token)
data_store = HuggingFaceObjectStore(DATA_REPO, repo_type="dataset", create=False, token=hub_token)
print(f"Hub account {run_store.whoami()} · runs {HUB_REPO} · data {DATA_REPO}")

In [ ]:
from hamic_jax.lab import complete_runs

finished = complete_runs(run_store)
if not finished:
    raise RuntimeError("No run has COMPLETE.json yet. Train or resume a run first.")
rows = [
    {
        "run_id": record.run_id,
        "step": record.step,
        "created": record.created,
        "model": record.manifest.get("preset", {}).get("name", ""),
        "parameters": record.manifest.get("param_count"),
        "total_steps": record.manifest.get("total_steps"),
    }
    for record in finished
]
for row in rows:
    print(row)

In [ ]:
from dataclasses import asdict

from IPython.display import display
from tqdm.auto import tqdm

from hamic_jax.lab import EvaluationRow, evaluate_completed_run
from hamic_jax.plots import evaluation_bars

selected_ids = RUN_IDS or [finished[0].run_id]
selected = [record for record in finished if record.run_id in selected_ids]
missing = set(selected_ids) - {record.run_id for record in selected}
if missing:
    raise KeyError(f"Unknown or incomplete runs: {sorted(missing)}")
selected_views = tuple(VIEWS)
if not selected_views:
    raise ValueError("Select at least one masking view.")

rows_out = []
evaluation_rows: list[EvaluationRow] = []
with tqdm(total=len(selected), desc="Evaluating finished runs", unit="run") as bar:
    for record in selected:
        bar.set_description(f"Evaluating {record.run_id}")
        evaluation = evaluate_completed_run(
            record,
            run_store,
            data_store,
            work_root,
            source_dir=source_dir,
            sequences=int(SEQUENCES),
            views=selected_views,
            mask_rate=float(MASK_RATE),
            log=print,
        )
        evaluation_rows.extend(evaluation)
        rows_out.extend(asdict(row) for row in evaluation)
        bar.update(1)

display(rows_out)
display(evaluation_bars(evaluation_rows))